# Optimized Hubbard–Holstein simulation

Cleaned and organized for reproducible execution from the repository root.


In [ ]:
## Plan (Pseudocode)
# 1. 导入库并设置参数
# 2. 构建算符与哈密顿量
# 3. 预先计算模拟块与数字块的指数算符
# 4. 初始态与精确演化（不变）
# 5. 使用预计算的指数算符进行 Trotter 演化并记录保真度
# 6. 测量总耗时并绘图

import numpy as np
import matplotlib.pyplot as plt
import time
from qutip import (
    tensor, basis, destroy, qeye,
    sigmaz, sigmax, sigmay,
    propagator, fidelity
)


In [ ]:
# 1. 系统参数
k = 1.0
U = 8.0 * k
g = 0.1 * k
omega0 = 8.0 * k
n_boson = 8
N_trotter = 200  # 增大步数
T_total = 1.0

dt = T_total / N_trotter

def tensor_ops(ops):
    out = ops[0]
    for o in ops[1:]:
        out = tensor(out, o)
    return out


In [ ]:
# 2. 构造算符
# Boson 模式
a1 = tensor(destroy(n_boson), qeye(n_boson), qeye(2), qeye(2), qeye(2), qeye(2))
a2 = tensor(qeye(n_boson), destroy(n_boson), qeye(2), qeye(2), qeye(2), qeye(2))
# 自旋 σ_z 列表
sig_z = sigmaz()
q_z = []
for i in range(4):
    ops = [qeye(n_boson), qeye(n_boson)] + [sig_z if j==i else qeye(2) for j in range(4)]
    q_z.append(tensor_ops(ops))

# Hamiltonian 各项
H_b   = omega0 * (a1.dag()*a1 + a2.dag()*a2)
H_ee  = U/4 * ((q_z[0]+q_z[1]) + q_z[0]*q_z[1]
             + (q_z[2]+q_z[3]) + q_z[2]*q_z[3])
H_ep  = g/2 * ((q_z[0]+q_z[1])*(a1.dag()+a1)
             + (q_z[2]+q_z[3])*(a2.dag()+a2))
# 跳跃项
pairs = [(1,2), (0,3)]
H_hop = 0
for i,j in pairs:
    # SxSx
    ops_x = [qeye(n_boson), qeye(n_boson)] + [sigmax() if idx in (i,j) else qeye(2) for idx in range(4)]
    H_hop += -k * tensor_ops(ops_x)
    # SySy
    ops_y = [qeye(n_boson), qeye(n_boson)] + [sigmay() if idx in (i,j) else qeye(2) for idx in range(4)]
    H_hop += -k * tensor_ops(ops_y)

H_analog = H_b + H_ee + H_ep
H_total  = H_analog + H_hop


In [ ]:
# 3. 预计算指数算符
U_analog  = (-1j * H_analog * dt).expm()
U_digital = (-1j * H_hop    * dt).expm()

In [ ]:
# 4. 初始态 & 精确演化
ferm = tensor(basis(2,0), basis(2,1), basis(2,1), basis(2,0))
boson = tensor(basis(n_boson,0), basis(n_boson,0))
psi0 = tensor(boson, ferm)

times   = np.linspace(0, T_total, N_trotter+1)
U_exact = [propagator(H_total, t) for t in times]
psi_ex  = [U_exact[i] * psi0 for i in range(len(times))]


In [ ]:
# 5. Trotter 演化 (仅矩阵-向量乘法)
psi = psi0
fid = [1.0]

t0 = time.time()
for n in range(1, N_trotter+1):
    psi = U_digital * (U_analog * psi)
    fid.append(fidelity(psi_ex[n], psi))
t1 = time.time()
print(f"Total simulation time: {t1-t0:.2f} s")


In [ ]:
# 6. 绘图
plt.ticklabel_format(useOffset=False, style='plain')  # 关闭刻度偏移
plt.figure()
plt.plot(times, fid, marker='o')
plt.xlabel('Time (1/k)')
plt.ylabel('Fidelity')
plt.title('DAQC Trotter Fidelity vs Exact (precomputed)')
plt.show()